In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Input, Conv2D, Lambda, Dense, Flatten, MaxPooling2D, Dropout
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K

In [2]:
# Step 1: Data Preparation - Function to Generate Image Pairs
def generate_image_pairs(image_directory, total_pairs=1000):
    categories = os.listdir(image_directory)
    categories = [c for c in categories if os.path.isdir(os.path.join(image_directory, c))]
    
    positive_pairs = []
    negative_pairs = []
    
    # Generate positive pairs
    for category in categories:
        path = os.path.join(image_directory, category)
        images = os.listdir(path)
        
        for i in range(int(total_pairs / (2 * len(categories)))):
            img1, img2 = np.random.choice(images, 2, replace=False)
            positive_pairs.append(([os.path.join(path, img1), os.path.join(path, img2)], 1))
    
    # Generate negative pairs
    for i in range(int(total_pairs / 2)):
        category1, category2 = np.random.choice(categories, 2, replace=False)
        path1, path2 = os.path.join(image_directory, category1), os.path.join(image_directory, category2)
        img1, img2 = np.random.choice(os.listdir(path1), 1)[0], np.random.choice(os.listdir(path2), 1)[0]
        negative_pairs.append(([os.path.join(path1, img1), os.path.join(path2, img2)], 0))
    
    pairs = positive_pairs + negative_pairs
    np.random.shuffle(pairs)
    
    return pairs

In [3]:
# Step 2: Preprocessing Image Pairs
def preprocess_pairs(pairs, target_size=(224, 224)):
    pair1 = []
    pair2 = []
    labels = []
    
    for pair, label in pairs:
        img1 = tf.keras.preprocessing.image.load_img(pair[0], target_size=target_size)
        img2 = tf.keras.preprocessing.image.load_img(pair[1], target_size=target_size)
        
        img1 = tf.keras.preprocessing.image.img_to_array(img1)
        img2 = tf.keras.preprocessing.image.img_to_array(img2)
        
        img1 /= 255.0
        img2 /= 255.0
        
        pair1.append(img1)
        pair2.append(img2)
        labels.append(label)
    
    return np.array(pair1), np.array(pair2), np.array(labels)

In [4]:
# Siamese Network Model Creation
def create_base_network(input_shape):
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    for layer in base_model.layers:
        layer.trainable = False
    model = Sequential([
        base_model,
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.5)
    ])
    return model

In [5]:
input_shape = (224, 224, 3)
base_network = create_base_network(input_shape)

In [6]:
input_a = Input(shape=input_shape)
input_b = Input(shape=input_shape)

In [7]:
processed_a = base_network(input_a)
processed_b = base_network(input_b)

In [8]:
distance = Lambda(lambda x: K.abs(x[0] - x[1]))([processed_a, processed_b])
outputs = Dense(1, activation='sigmoid')(distance)
model = Model([input_a, input_b], outputs)

In [9]:
model.compile(loss='binary_crossentropy', optimizer=Adam(0.0001), metrics=['accuracy'])

In [10]:
# Model Training
def train_model(image_directory, total_pairs=1000, epochs=10):
    pairs = generate_image_pairs(image_directory, total_pairs)
    pair1, pair2, labels = preprocess_pairs(pairs)
    history = model.fit([pair1, pair2], labels, epochs=epochs, batch_size=32)
    return history

In [11]:
# Prediction Function
def predict_member_similarity(model, img_path1, img_path2, target_size=(224, 224)):
    img1 = tf.keras.preprocessing.image.load_img(img_path1, target_size=target_size)
    img2 = tf.keras.preprocessing.image.load_img(img_path2, target_size=target_size)
    
    img1 = tf.keras.preprocessing.image.img_to_array(img1) / 255.0
    img2 = tf.keras.preprocessing.image.img_to_array(img2) / 255.0
    
    pred = model.predict([np.array([img1]), np.array([img2])])
    return pred[0][0]

In [12]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# Callbacks
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, verbose=1)

# Assuming you have a directory with training images organized in subdirectories for each class
image_directory = "C:\\Users\\dhars\\Desktop\\BTS_1\\Training Data"

# Generate image pairs
pairs = generate_image_pairs(image_directory)

# Preprocess image pairs
pair1, pair2, labels = preprocess_pairs(pairs)

# Now proceed to training with these variables
# Reduced epochs and increased batch size for quicker training
history = model.fit(
    [pair1, pair2], labels,
    validation_split=0.1,
    epochs=70,  # Reduced from 70 to 20
    batch_size=32,  # Increased from 32 to 64
    callbacks=[reduce_lr, early_stopping]
)

Epoch 1/70


29/29 [==============================] - 598s 20s/step - loss: 0.8202 - accuracy: 0.4894 - val_loss: 0.6911 - val_accuracy: 0.4900 - lr: 1.0000e-04
Epoch 2/70
29/29 [==============================] - 515s 18s/step - loss: 0.7317 - accuracy: 0.5206 - val_loss: 0.6941 - val_accuracy: 0.5000 - lr: 1.0000e-04
Epoch 3/70
29/29 [==============================] - 381s 13s/step - loss: 0.7390 - accuracy: 0.4939 - val_loss: 0.6862 - val_accuracy: 0.5500 - lr: 1.0000e-04
Epoch 4/70
29/29 [==============================] - 413s 14s/step - loss: 0.7180 - accuracy: 0.4939 - val_loss: 0.6955 - val_accuracy: 0.5300 - lr: 1.0000e-04
Epoch 5/70
29/29 [==============================] - 400s 14s/step - loss: 0.6963 - accuracy: 0.5530 - val_loss: 0.6909 - val_accuracy: 0.5600 - lr: 1.0000e-04
Epoch 6/70
29/29 [==============================] - ETA: 0s - loss: 0.6965 - accuracy: 0.5362 
Epoch 6: ReduceLROnPlateau reducing learning rate to 1.9999999494757503e-05.
29/29 [========================

In [13]:
# Assuming you have a function or process to generate test pairs similar to the training pairs
test_pairs = generate_image_pairs("C:\\Users\\dhars\\Desktop\\BTS_1\\Testing Data", total_pairs=200)
test_pair1, test_pair2, test_labels = preprocess_pairs(test_pairs)

# Evaluate the model on the test set
evaluation = model.evaluate([test_pair1, test_pair2], test_labels)
print(f"Test Loss: {evaluation[0]}, Test Accuracy: {evaluation[1]}")

7/7 [==============================] - 76s 11s/step - loss: 0.6934 - accuracy: 0.4899
Test Loss: 0.6933881044387817, Test Accuracy: 0.4898989796638489


In [14]:
# Example usage of the prediction function
similarity_score = predict_member_similarity(model, "C:\\Users\\dhars\\Desktop\\Reference Images\\Jin\\member.jpg", "C:\\Users\\dhars\\Desktop\\New folder\\bhblhkbh.jpeg")
print(f"Similarity Score: {similarity_score}")

1/1 [==============================] - 1s 1s/step
Similarity Score: 0.5122614502906799


In [15]:
# # Save the model
# model.save("C:\\Users\\dhars\\Desktop\\bts_siamese_model.h5")

# Load the model
model = tf.keras.models.load_model('bts_siamese_model.h5')

In [16]:
from sklearn.metrics import classification_report, confusion_matrix

# Predictions
predictions = model.predict([test_pair1, test_pair2])
predicted_labels = [1 if pred > 0.5 else 0 for pred in predictions]

# True labels
true_labels = test_labels

# Classification report
print(classification_report(true_labels, predicted_labels))

# Confusion matrix
conf_matrix = confusion_matrix(true_labels, predicted_labels)
print(conf_matrix)


7/7 [==============================] - 91s 13s/step
              precision    recall  f1-score   support

           0       0.44      0.59      0.50       100
           1       0.36      0.23      0.28        98

    accuracy                           0.41       198
   macro avg       0.40      0.41      0.39       198
weighted avg       0.40      0.41      0.40       198

[[59 41]
 [75 23]]


In [17]:
# import ipywidgets as widgets
# from IPython.display import display, Image
# from tensorflow.keras.preprocessing import image as keras_image

# # Load the model
# model = tf.keras.models.load_model('bts_siamese_model.h5')

# # Directory containing one reference image per BTS member
# reference_images_dir = "C:\\Users\\dhars\\Desktop\\Reference Images\\Jin"

# # Load reference images and preprocess them
# reference_images = {}
# for filename in os.listdir(reference_images_dir):
#     member_name = os.path.splitext(filename)[0]  # Assuming the filename is the member's name
#     img_path = os.path.join(reference_images_dir, filename)
#     img = keras_image.load_img(img_path, target_size=(224, 224))
#     img_array = keras_image.img_to_array(img) / 255.0
#     reference_images[member_name] = img_array

# # Define the prediction function
# def predict_member(upload_image_path):
#     img = keras_image.load_img(upload_image_path, target_size=(224, 224))
#     upload_img_array = keras_image.img_to_array(img) / 255.0
    
#     highest_similarity = 0
#     predicted_member = None
    
#     # Compare the uploaded image with each reference image
#     for member, ref_img_array in reference_images.items():
#         similarity = model.predict([np.array([upload_img_array]), np.array([ref_img_array])])[0][0]
#         if similarity > highest_similarity:
#             highest_similarity = similarity
#             predicted_member = member
    
#     return predicted_member, highest_similarity

# # Upload button
# uploader = widgets.FileUpload()
# display(uploader)

# # Prediction button
# predict_button = widgets.Button(description="Predict Member")

# def on_predict_button_clicked(b):
#     # Check if a file has been uploaded
#     if uploader.value:
#         # Grab uploaded file data
#         uploaded_file_data = next(iter(uploader.value.values()))
#         file_content = uploaded_file_data['content']
        
#         # Write to a local file
#         with open("uploaded_image.jpeg", "wb") as f:
#             f.write(file_content)
        
#         # Display the uploaded image
#         display(Image("uploaded_image.jpeg"))
        
#         # Predict the member
#         predicted_member, confidence = predict_member("uploaded_image.jpeg")
#         print(f"Predicted Member: {predicted_member} with confidence: {confidence:.2%}")
#     else:
#         print("Please upload an image.")
#     print(uploader.value)
        
# predict_button.on_click(on_predict_button_clicked)
# display(predict_button)


In [18]:
import ipywidgets as widgets
from IPython.display import display, Image
from tensorflow.keras.preprocessing import image as keras_image
import os
import numpy as np

# Load the model
model = tf.keras.models.load_model('bts_siamese_model.h5')

# Directory containing one reference image per BTS member
reference_images_dir = 'C:\\Users\\dhars\\Desktop\\Reference Images\\Jin'

# Load reference images and preprocess them
reference_images = {}
for filename in os.listdir(reference_images_dir):
    member_name = os.path.splitext(filename)[0]  # Assuming the filename is the member's name
    img_path = os.path.join(reference_images_dir, filename)
    img = keras_image.load_img(img_path, target_size=(224, 224))
    img_array = keras_image.img_to_array(img) / 255.0
    reference_images[member_name] = img_array

# Define the prediction function
def predict_member(upload_image_array):
    highest_similarity = 0
    predicted_member = None
    
    # Compare the uploaded image with each reference image
    for member, ref_img_array in reference_images.items():
        similarity = model.predict([np.array([upload_image_array]), np.array([ref_img_array])])[0][0]
        if similarity > highest_similarity:
            highest_similarity = similarity
            predicted_member = member
    
    return predicted_member, highest_similarity

# Upload button
uploader = widgets.FileUpload()
display(uploader)

# Prediction button
predict_button = widgets.Button(description="Predict Member")
display(predict_button)

def on_predict_button_clicked(b):
    if uploader.value:
        # Get the uploaded file data
        # Since uploader.value is misbehaving, access the data directly
        uploaded_file = list(uploader.value.items())[0]
        file_name, file_info = uploaded_file
        content = file_info['content']
        with open("uploaded_image.jpeg", "wb") as f:
            f.write(content)
        
        # Display the uploaded image
        display(Image("uploaded_image.jpeg"))
        
        # Preprocess the image and make predictions
        img = keras_image.load_img("uploaded_image.jpeg", target_size=(224, 224))
        img_array = keras_image.img_to_array(img) / 255.0

        predicted_member, confidence = predict_member(img_array)
        print(f"Predicted Member: {predicted_member} with confidence: {confidence:.2%}")
    else:
        print("Please upload an image.")


# Connect the button to the function
predict_button.on_click(on_predict_button_clicked)


FileUpload(value=(), description='Upload')

Button(description='Predict Member', style=ButtonStyle())